In [6]:
import cv2
import time
import random
import numpy as np
import urllib.request
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [4]:
url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
urllib.request.urlretrieve(url, "face_landmarker.task")
print("Archivo face_landmarker.task descargado correctamente.")

Archivo face_landmarker.task descargado correctamente.


In [13]:
W, H = 640, 480
OBJETOS = ["GAFAS_THUG", "CORONA", "PAYASO", "ALIEN"]
BASE_X, BASE_Y = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))

detector = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path='face_landmarker.task'),
        output_face_blendshapes=True,
        num_faces=1
    )
)


def lente_radial(map_x, map_y, centro, radio, potencia):
    """Expande (potencia > 0) o encoge (potencia < 0) un círculo alrededor de 'centro'."""
    dx, dy = map_x - centro[0], map_y - centro[1]
    dist = np.hypot(dx, dy)
    mask = dist < radio
    nd = dist[mask] / radio
    escala = (nd ** (1.0 + potencia)) / (nd + 1e-6)
    map_x[mask] = centro[0] + dx[mask] * escala
    map_y[mask] = centro[1] + dy[mask] * escala


def aplicar_filtro(frame, nariz, ojos, dist_ojos, modo, intensidad):
    """Aplica deformaciones espaciales y ambientación de color según la expresión."""
    if modo == "NORMAL" or intensidad <= 0.02:
        return frame

    map_x, map_y = BASE_X.copy(), BASE_Y.copy()
    radio = dist_ojos * 2.15

    if modo == "SONRISA":
        lente_radial(map_x, map_y, nariz, radio, intensidad * 0.95)
        frame = cv2.remap(frame, map_x, map_y, cv2.INTER_LINEAR)
        overlay = np.full_like(frame, (30, 140, 255))
        for s in (-1, 1):
            cv2.circle(overlay, (int(nariz[0] + s * dist_ojos * 0.65), int(nariz[1] + dist_ojos * 0.25)),
                       max(10, int(dist_ojos * 0.28)), (130, 110, 255), -1)
        return cv2.addWeighted(overlay, 0.16 * intensidad, frame, 1.0 - 0.16 * intensidad, 0)

    if modo == "CENO":
        lente_radial(map_x, map_y, nariz, radio, -intensidad * 0.58)
        frame = cv2.remap(frame, map_x, map_y, cv2.INTER_LINEAR)
        gris = cv2.cvtColor(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
        frame = cv2.addWeighted(gris, 0.45 * intensidad, frame, 1.0 - 0.45 * intensidad, 0)
        vineta = np.clip(1.15 - (np.hypot(BASE_X - nariz[0], BASE_Y - nariz[1]) / (W * 0.65)) * 0.65 * intensidad, 0.35, 1.0)
        return (frame * vineta[..., None]).astype(np.uint8)

    if modo == "SORPRESA":
        for ojo in ojos:
            lente_radial(map_x, map_y, ojo, dist_ojos * 0.52, intensidad * 1.15)
        frame = cv2.remap(frame, map_x, map_y, cv2.INTER_LINEAR)
        if intensidad > 0.45:
            for ang in np.radians(range(0, 360, 15)):
                p1 = (int(W / 2 + np.cos(ang) * W * 0.42), int(H / 2 + np.sin(ang) * H * 0.42))
                p2 = (int(W / 2 + np.cos(ang) * W * 0.75), int(H / 2 + np.sin(ang) * H * 0.75))
                cv2.line(frame, p1, p2, (255, 255, 255), 2)
    return frame


def dibujar_objeto(frame, centro, u, angulo, tipo):
    """Renderiza el objeto AR con sombreado 3D, sombra proyectada y rotación suave."""
    tam = int(u * 5.6)
    if not tipo or tam < 20:
        return frame

    c = tam // 2
    obj = np.zeros((tam, tam, 4), dtype=np.uint8)
    pt = lambda dx, dy: (int(c + dx * u), int(c + dy * u))
    grosor = max(2, int(u * 0.06))

    if tipo == "GAFAS_THUG":
        # Matriz pixel-art (mitad derecha, columnas 0..14) para gafas 8-bit nítidas
        mitad = [
            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
            [0, 0, 1, 2, 3, 2, 2, 3, 2, 2, 2, 2, 1, 0, 0],
            [0, 0, 1, 2, 2, 3, 2, 2, 3, 2, 2, 2, 1, 0, 0],
            [0, 0, 1, 2, 2, 2, 4, 2, 2, 4, 2, 2, 1, 0, 0],
            [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        ]
        paleta = {
            1: (15, 15, 20, 255),     # Montura negra
            2: (40, 42, 52, 235),     # Cristal oscuro semitransparente
            3: (255, 255, 255, 255),  # Reflejo blanco
            4: (180, 190, 205, 245),  # Brillo suave inferior
        }
        px = max(2, int(u * 0.086))
        y0 = int(c - 2.3 * px)
        for r, fila in enumerate(mitad):
            for col, val in enumerate(fila):
                if val == 0:
                    continue
                for s in (-1, 1):
                    if col == 0 and s == -1:
                        continue
                    x_px = c + s * col * px - px // 2
                    y_px = y0 + r * px
                    cv2.rectangle(obj, (x_px, y_px), (x_px + px, y_px + px), paleta[val], -1)

    elif tipo == "CORONA":
        # Fondo interior de terciopelo carmesí para dar profundidad 3D
        terciopelo = np.array([pt(-0.78, -1.10), pt(-0.65, -1.65), pt(0, -1.85),
                               pt(0.65, -1.65), pt(0.78, -1.10)], np.int32)
        cv2.fillPoly(obj, [terciopelo], (30, 15, 130, 255), cv2.LINE_AA)

        # Cuerpo dorado principal + faceta central iluminada
        corona = np.array([
            pt(-0.88, -1.02), pt(-1.02, -1.72), pt(-0.70, -1.36), pt(-0.48, -1.88),
            pt(-0.22, -1.40), pt(0.00, -2.04), pt(0.22, -1.40), pt(0.48, -1.88),
            pt(0.70, -1.36), pt(1.02, -1.72), pt(0.88, -1.02)
        ], np.int32)
        cv2.fillPoly(obj, [corona], (15, 180, 245, 255), cv2.LINE_AA)

        brillo_central = np.array([
            pt(-0.34, -1.02), pt(-0.22, -1.40), pt(0.00, -2.04), pt(0.22, -1.40), pt(0.34, -1.02)
        ], np.int32)
        cv2.fillPoly(obj, [brillo_central], (65, 225, 255, 255), cv2.LINE_AA)
        cv2.polylines(obj, [corona], True, (10, 115, 175, 255), grosor, cv2.LINE_AA)

        # Diadema inferior con perlas
        cv2.rectangle(obj, pt(-0.88, -1.16), pt(0.88, -0.98), (15, 155, 225, 255), -1)
        cv2.rectangle(obj, pt(-0.88, -1.16), pt(0.88, -0.98), (10, 110, 170, 255), max(1, grosor - 1))
        for px_rel in np.linspace(-0.72, 0.72, 7):
            cv2.circle(obj, pt(px_rel, -1.07), max(2, int(u * 0.04)), (245, 250, 255, 255), -1, cv2.LINE_AA)

        # Bolas doradas en las puntas
        for tip in [(-1.02, -1.72), (-0.48, -1.88), (0.00, -2.04), (0.48, -1.88), (1.02, -1.72)]:
            r_tip = max(3, int(u * 0.085))
            cv2.circle(obj, pt(*tip), r_tip, (30, 210, 255, 255), -1, cv2.LINE_AA)
            cv2.circle(obj, pt(*tip), r_tip, (10, 115, 175, 255), 1, cv2.LINE_AA)

        # Gemas facetadas (rubí central, zafiro y esmeralda laterales)
        rubi = np.array([pt(0, -1.55), pt(0.13, -1.40), pt(0, -1.25), pt(-0.13, -1.40)], np.int32)
        cv2.fillPoly(obj, [rubi], (45, 30, 235, 255), cv2.LINE_AA)
        cv2.polylines(obj, [rubi], True, (20, 10, 140, 255), 2, cv2.LINE_AA)
        cv2.circle(obj, pt(-0.03, -1.44), max(2, int(u * 0.03)), (255, 255, 255, 255), -1)

        for gx, col in [(-0.54, (230, 150, 20)), (0.54, (50, 200, 60))]:
            rg = max(4, int(u * 0.09))
            cv2.circle(obj, pt(gx, -1.35), rg, (*col, 255), -1, cv2.LINE_AA)
            cv2.circle(obj, pt(gx, -1.35), rg, (20, 60, 80, 255), 1, cv2.LINE_AA)
            cv2.circle(obj, pt(gx - 0.03, -1.38), max(1, rg // 3), (255, 255, 255, 240), -1)

    elif tipo == "PAYASO":
        # Nariz 3D esférica con degradado radial y reflejo brillante
        nr = max(9, int(u * 0.33))
        nc = pt(0, 0.50)
        cv2.circle(obj, nc, nr, (15, 15, 165, 255), -1, cv2.LINE_AA)
        cv2.circle(obj, (nc[0] - int(nr * 0.12), nc[1] - int(nr * 0.12)), int(nr * 0.84), (30, 30, 230, 255), -1, cv2.LINE_AA)
        cv2.circle(obj, (nc[0] - int(nr * 0.22), nc[1] - int(nr * 0.22)), int(nr * 0.52), (65, 95, 255, 255), -1, cv2.LINE_AA)
        cv2.ellipse(obj, (nc[0] - int(nr * 0.34), nc[1] - int(nr * 0.34)),
                    (max(2, int(nr * 0.26)), max(1, int(nr * 0.14))), -35, 0, 360, (255, 255, 255, 240), -1, cv2.LINE_AA)

        # Gorro cónico con base curva y lunares recortados
        gorro_capa = np.zeros_like(obj)
        cono = np.array([pt(-0.68, -1.02), pt(0.68, -1.02), pt(0, -2.26)], np.int32)
        cv2.fillPoly(gorro_capa, [cono], (225, 75, 155, 255), cv2.LINE_AA)
        cv2.ellipse(gorro_capa, pt(0, -1.02), (max(10, int(0.68 * u)), max(4, int(0.15 * u))),
                    0, 0, 180, (225, 75, 155, 255), -1, cv2.LINE_AA)

        # Lunares de colores solo dentro de la silueta del gorro
        mascara_gorro = gorro_capa[:, :, 3] > 0
        lunares_capa = gorro_capa.copy()
        for lx, ly, col in [(-0.22, -1.25, (0, 230, 255)), (0.24, -1.22, (100, 255, 120)),
                            (0.0, -1.52, (255, 230, 80)), (-0.14, -1.82, (80, 180, 255)), (0.14, -1.78, (0, 230, 255))]:
            cv2.circle(lunares_capa, pt(lx, ly), max(3, int(u * 0.085)), (*col, 255), -1, cv2.LINE_AA)
        gorro_capa[mascara_gorro] = lunares_capa[mascara_gorro]
        obj = np.maximum(obj, gorro_capa)

        # Ribete fruncido inferior y pompón esponjoso superior
        for rx in np.linspace(-0.64, 0.64, 9):
            ry = -1.02 + 0.12 * np.cos(rx * np.pi / 1.36)
            cv2.circle(obj, pt(rx, ry), max(4, int(u * 0.11)), (245, 250, 255, 255), -1, cv2.LINE_AA)
            cv2.circle(obj, pt(rx, ry), max(4, int(u * 0.11)), (190, 200, 215, 255), 1, cv2.LINE_AA)

        rp = max(6, int(u * 0.16))
        for ang in np.radians(range(0, 360, 60)):
            cv2.circle(obj, (int(c + np.cos(ang) * rp * 0.65), int(c - 2.30 * u + np.sin(ang) * rp * 0.65)),
                       int(rp * 0.72), (0, 215, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(obj, pt(0, -2.30), int(rp * 0.8), (80, 245, 255, 255), -1, cv2.LINE_AA)

    elif tipo == "ALIEN":
        t = np.linspace(0, 1, 18)[:, None]
        for s in (-1, 1):
            # Base orgánica sobre la cabeza
            cv2.ellipse(obj, pt(s * 0.42, -1.00), (max(5, int(0.16 * u)), max(3, int(0.08 * u))),
                        s * 12, 0, 360, (30, 150, 40, 255), -1, cv2.LINE_AA)

            # Tallo curvo Bézier
            p0, p1, p2 = np.array([s * 0.42, -1.00]), np.array([s * 0.96, -1.48]), np.array([s * 0.76, -1.98])
            curva = (1 - t) ** 2 * p0 + 2 * (1 - t) * t * p1 + t ** 2 * p2
            pts_curva = np.array([pt(x, y) for x, y in curva], np.int32)
            cv2.polylines(obj, [pts_curva], False, (20, 115, 30, 255), max(4, int(u * 0.11)), cv2.LINE_AA)
            cv2.polylines(obj, [pts_curva], False, (75, 235, 95, 255), max(2, int(u * 0.06)), cv2.LINE_AA)

            # Esfera bioluminiscente con aura neón
            tip = pt(s * 0.76, -1.98)
            ro = max(7, int(u * 0.21))
            cv2.circle(obj, tip, int(ro * 1.55), (90, 255, 130, 75), -1, cv2.LINE_AA)
            cv2.circle(obj, tip, ro, (55, 235, 95, 255), -1, cv2.LINE_AA)
            cv2.circle(obj, tip, int(ro * 0.65), (165, 255, 190, 255), -1, cv2.LINE_AA)
            cv2.circle(obj, (tip[0] - int(ro * 0.28), tip[1] - int(ro * 0.28)),
                       max(2, int(ro * 0.24)), (255, 255, 255, 245), -1, cv2.LINE_AA)

    # Generar sombra proyectada suave debajo del objeto
    dy_sombra = max(2, int(u * 0.06))
    sombra_alfa = cv2.GaussianBlur(obj[:, :, 3], (0, 0), max(1.5, u * 0.05)) * 0.42
    lienzo = np.zeros_like(obj)
    lienzo[dy_sombra:, :, 3] = sombra_alfa[:-dy_sombra].astype(np.uint8)

    # Combinar objeto nítido encima de su sombra
    a_obj = obj[:, :, 3:4] / 255.0
    a_som = lienzo[:, :, 3:4] / 255.0
    a_total = a_obj + a_som * (1.0 - a_obj)
    lienzo[:, :, :3] = (obj[:, :, :3] * a_obj).astype(np.uint8)
    lienzo[:, :, 3:4] = (a_total * 255.0).astype(np.uint8)

    # Rotar al ángulo de la cabeza y mezclar con el fotograma
    rotado = cv2.warpAffine(lienzo, cv2.getRotationMatrix2D((c, c), angulo, 1.0), (tam, tam))
    x1, y1 = centro[0] - c, centro[1] - c
    fx1, fy1, fx2, fy2 = max(0, x1), max(0, y1), min(W, x1 + tam), min(H, y1 + tam)
    if fx1 < fx2 and fy1 < fy2:
        rec = rotado[fy1 - y1:fy2 - y1, fx1 - x1:fx2 - x1]
        alfa = rec[:, :, 3:4] / 255.0
        frame[fy1:fy2, fx1:fx2] = (alfa * rec[:, :, :3] + (1.0 - alfa) * frame[fy1:fy2, fx1:fx2]).astype(np.uint8)
    return frame


cap = cv2.VideoCapture(0)
mostrar_malla, modo_activo, intensidad_suave = False, "NORMAL", 0.0
inicio_cierre, sorteado, objeto_actual, fin_objeto = None, False, None, 0.0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(cv2.resize(frame, (W, H)), 1)
    ahora = time.time()
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

    if res.face_landmarks and res.face_blendshapes:
        lm = res.face_landmarks[0]
        bs = {b.category_name: b.score for b in res.face_blendshapes[0]}
        medio = lambda i, j: (int((lm[i].x + lm[j].x) * W / 2), int((lm[i].y + lm[j].y) * H / 2))

        nariz = (int(lm[1].x * W), int(lm[1].y * H))
        ojo_izq, ojo_der = medio(33, 133), medio(362, 263)
        centro_ojos = ((ojo_izq[0] + ojo_der[0]) // 2, (ojo_izq[1] + ojo_der[1]) // 2)
        dist_ojos = np.hypot(ojo_der[0] - ojo_izq[0], ojo_der[1] - ojo_izq[1])
        angulo = -np.degrees(np.arctan2(ojo_der[1] - ojo_izq[1], ojo_der[0] - ojo_izq[0]))

        # Detección de ojos cerrados (1.5s) para sortear objeto AR durante 5s
        if bs.get('eyeBlinkLeft', 0) > 0.45 and bs.get('eyeBlinkRight', 0) > 0.45:
            inicio_cierre = inicio_cierre or ahora
            if ahora - inicio_cierre >= 1.5 and not sorteado:
                objeto_actual = random.choice([o for o in OBJETOS if o != objeto_actual])
                fin_objeto, sorteado = ahora + 5.0, True
        else:
            inicio_cierre, sorteado = None, False
        if ahora > fin_objeto:
            objeto_actual = None

        # Expresiones faciales
        sonrisa = (bs.get('mouthSmileLeft', 0) + bs.get('mouthSmileRight', 0)) / 2
        ceno = (bs.get('browDownLeft', 0) + bs.get('browDownRight', 0)) / 2
        sorpresa = max(bs.get('browInnerUp', 0), bs.get('jawOpen', 0) * 0.9)

        modo_obj, int_obj = "NORMAL", 0.0
        if sorpresa > 0.48:
            modo_obj, int_obj = "SORPRESA", min(1.0, (sorpresa - 0.30) * 1.4)
        elif sonrisa > 0.42:
            modo_obj, int_obj = "SONRISA", min(1.0, (sonrisa - 0.25) * 1.35)
        elif ceno > 0.38:
            modo_obj, int_obj = "CENO", min(1.0, (ceno - 0.20) * 1.5)

        if modo_obj != "NORMAL":
            modo_activo = modo_obj
        intensidad_suave = 0.72 * intensidad_suave + 0.28 * int_obj

        frame = dibujar_objeto(frame, centro_ojos, dist_ojos, angulo, objeto_actual)
        
        if mostrar_malla:
            for p in lm[::3]:
                cv2.circle(frame, (int(p.x * W), int(p.y * H)), 1, (0, 255, 180), -1)
        
        frame = aplicar_filtro(frame, nariz, [ojo_izq, ojo_der], dist_ojos, modo_activo, intensidad_suave)


    cv2.imshow("Filtro Interactivo", frame)
    tecla = cv2.waitKey(1) & 0xFF
    if tecla == 27:
        break
    if tecla in (ord('l'), ord('L')):
        mostrar_malla = not mostrar_malla

cap.release()
cv2.destroyAllWindows()